In [1]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [2]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [3]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)

voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression(random_state=42)),
                             ('rf', RandomForestClassifier(random_state=42)),
                             ('svc', SVC(random_state=42))])

In [4]:
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [ ]:
# this means that the predicted output belongs to class 1
voting_clf.predict(X_test[:2])

array([1, 0])

In [7]:
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]

[array([1]), array([1]), array([0])]

In [8]:
voting_clf.score(X_test, y_test)

0.912

### Soft Voting

In [10]:
voting_clf.voting = 'soft'
voting_clf.named_estimators['svc'].probability = True
voting_clf.fit(X_train, y_train)
voting_clf.score(X_test, y_test)

0.92

### Bagging and Pasting

In [11]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1, random_state=42)

### Out-of-Bag Evaluation

In [12]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, oob_score=True, max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.9253333333333333

In [13]:
from sklearn.metrics import accuracy_score
y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.904

### Random Forests

In [14]:
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)
y_pred_rf = rnd_clf.predict(X_test)

In [15]:
# this is the same with the codes above:
bag_clf=BaggingClassifier(DecisionTreeClassifier(max_features='sqrt', max_leaf_nodes=16), n_estimators=500, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)

### Adaptive Boosting

In [16]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=30, learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=30, random_state=42)

# Exercises

1. If you have trained five different models on the exact same training data, and they all achieve 95% precision, is there any chance that you can combine these models to get better results? If so, how? If not, why?
   - Yes, there is a chance that I can combine these models. I can probably consider a way on how to choose the best predicted value out of all the five models' predictions using different ways -- either through soft or hard voting. In this way, we aggregate the predicted values to find the best predicted value. There are also other ways to enhance the performance of the predicted outcomes: either through boosting or stacking. 
2. What is the difference between hard and soft voting classifiers?
   - Hard voting classifiers rely on the predicted outputs while soft voting classifiers consider the probability of each of the outcomes.
3. Is it possible to speed up training of a bagging ensemble by distributing it across multiple servers? What about pasting ensembles, boosting ensembles, random forests, or stacking ensembles?
    - Yes, it is possible to speed up training of a bagging ensemble by distributing it across multiple servers. 
    - Pasting ensemble -> yes
    - Boosting ensembles -> no
    - Random forests -> yes
    - Stacking ensembles -> no
4. What is the benefit of out-of-bag evaluation?
    - Out of bag evaluation allows a bagging ensemble to estimate its generalization performance without requiring a separate validation set.
5. What makes extra-trees ensembles more random than regular random forests? How can this extra randomness help? Are extra-trees classifiers slower or faster than regular random forests?
    - Instead of using thresholds, it chooses random thresholds. This extra randomness helps since it allows to pick the best split among those random choices. This is faster than regular random forests.
6. If your AdaBoost ensemble underfits the training data, which hyperparameters should you tweak, and how?
    - Increasing n_estimators or decreasing the regularization of the base estimator.
7. If your gradient boosting ensemble overfits the training set, should you increase or decrease the learning rate?
   - Decrease the learning rate so that it will generalize well. 

8. Load the MNIST dataset (introduced in Chapter 3), and split it into a training set, a validation set, and a test set (e.g., use 50,000 instances for training, 10,000 for validation, and 10,000 for testing). Then train various classifiers, such as a random forest classifier, an extra-trees classifier, and an SVM classifier. Next, try
to combine them into an ensemble that outperforms each individual classifier on the validation set, using soft or hard voting. Once you have found one, try it on the test set. How much better does it perform compared to the individual classifiers?

In [4]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', as_frame=False)
X, y = mnist.data, mnist.target

In [6]:
from sklearn.model_selection import train_test_split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, train_size=60000, test_size=10000, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=10000, random_state=42)

In [17]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(50000, 784)
(10000, 784)
(10000, 784)


In [19]:
# random forest classifier
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)


RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)

In [18]:
# extra-trees classifier
from sklearn.ensemble import ExtraTreesClassifier
xtra_clf = ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=42)
xtra_clf.fit(X_train, y_train)

ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=42)

In [16]:
from sklearn.svm import SVC
svc = SVC(random_state = 42)
svc.fit(X_train, y_train)

SVC()

In [20]:
# to determine first the performance of each of the model in the validation set
from sklearn.metrics import accuracy_score

rnd_clf_pred = rnd_clf.predict(X_val)
xtra_clf_pred = xtra_clf.predict(X_val)
svc_pred = svc.predict(X_val)

print(f"Random Forest Classifier Score: {accuracy_score(y_val, rnd_clf_pred)}")
print(f"Extra Trees Classifier Score: {accuracy_score(y_val, xtra_clf_pred)}")
print(f"SVM Classifier Score: {accuracy_score(y_val, svc_pred)}")


Random Forest Classifier Score: 0.9711
Extra Trees Classifier Score: 0.973
SVM Classifier Score: 0.9788


In [ ]:
# creating the hard voting classifier through ensemble learning
from sklearn.ensemble import VotingClassifier
voting_clf = VotingClassifier(
    estimators=[
        ('randf_clf', rnd_clf),
        ('xtrees_clf', xtra_clf),
        ('svc_clf', svc)
    ]
)
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('randf_clf',
                              RandomForestClassifier(n_estimators=500,
                                                     random_state=42)),
                             ('xtrees_clf',
                              ExtraTreesClassifier(n_estimators=500,
                                                   random_state=42)),
                             ('svc_clf', SVC(random_state=42))])

In [27]:
voting_clf_pred = voting_clf.predict(X_val)
print(f"Voting Classifier Score: {accuracy_score(y_val, voting_clf_pred)}")

Voting Classifier Score: 0.974


In [29]:
# since hard voting classifier does not outperform all, we go with soft voting

svc_soft = SVC(probability=True, random_state=42)
soft_voting_clf = VotingClassifier(
    estimators=[
        ('randf_clf', rnd_clf),
        ('xtrees_clf', xtra_clf),
        ('svc_clf', svc_soft)
    ],
    voting='soft'
)
soft_voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('randf_clf',
                              RandomForestClassifier(n_estimators=500,
                                                     n_jobs=-1,
                                                     random_state=42)),
                             ('xtrees_clf',
                              ExtraTreesClassifier(n_estimators=500, n_jobs=-1,
                                                   random_state=42)),
                             ('svc_clf',
                              SVC(probability=True, random_state=42))],
                 voting='soft')

In [30]:
soft_voting_clf_pred = soft_voting_clf.predict(X_val)
print(f"Soft Voting Classifier Score: {accuracy_score(y_val, soft_voting_clf_pred)}")

Soft Voting Classifier Score: 0.9796


### We found out that soft voting outperforms each individual classifier, we now get the scores for the test set.

In [ ]:
rnd_clf_pred_test = rnd_clf.predict(X_test)
xtra_clf_pred_test = xtra_clf.predict(X_test)
svc_pred_test = svc.predict(X_test)
soft_voting_clf_pred_test = soft_voting_clf.predict(X_test)


print(f"Random Forest Classifier Final Score: {accuracy_score(y_test, rnd_clf_pred_test)}")
print(f"Extra Trees Classifier Final Score: {accuracy_score(y_test, xtra_clf_pred_test)}")
print(f"SVM Classifier Final Score: {accuracy_score(y_test, svc_pred_test)}")
print(f"Soft Voting Classifier Final Score: {accuracy_score(y_test, soft_voting_clf_pred_test)}")

Random Forest Classifier Final Score: 0.966
Extra Trees Classifier Final Score: 0.9705
SVM Classifier Final Score: 0.976
Soft Voting Classifier Final Score: 0.9764


The soft voting classifier achieved the best validation accuracy at $97.96%$. On the test set, it achieved $97.64%$ accuracy, slightly outperforming the best individual classifier, the SVM, which achieved $97.60%$. The ensemble therefore improved test accuracy by $0.04$ percentage points over the best individual classifier.